# Unidad 3 · Colab 1 de 3
## Fundamentos de bases de datos relacionales y SQL

**Objetivos de este notebook**

- Entender el modelo relacional: tablas, filas, columnas, claves primarias y foráneas.
- Comparar SQLite y PostgreSQL, y cuándo usar cada uno.
- Escribir SQL: `SELECT`, `INSERT`, `UPDATE`, `DELETE` y distintos tipos de `JOIN`.
- Practicar todo esto con un mini esquema de e-commerce, que vamos a retomar en el Colab 2 con SQLAlchemy.

> **Nivel:** intermedio. Se asume Python básico; no hace falta experiencia previa con SQL.

**Cómo practicamos en Colab:** usamos el módulo `sqlite3` de la librería estándar de Python — no requiere instalar nada ni levantar un servidor.

---

## 1. El modelo relacional

Una base de datos relacional organiza los datos en **tablas** (filas y columnas). Cada fila es un registro, cada columna un atributo. Conceptos clave:

- **Clave primaria (PRIMARY KEY):** identifica de forma única cada fila de una tabla.
- **Clave foránea (FOREIGN KEY):** una columna que referencia la clave primaria de otra tabla, modelando una relación entre ambas.
- **Normalización:** organizar las tablas para evitar datos duplicados o inconsistentes (por ejemplo, el nombre de un cliente vive una sola vez en `cliente`, no repetido en cada pedido).

Para este notebook vamos a usar tres tablas de un e-commerce simplificado: `cliente`, `producto` y `pedido` (con `cliente_id` como clave foránea).

## 2. SQLite vs. PostgreSQL

| | SQLite | PostgreSQL |
|---|---|---|
| Dónde vive | Un único archivo .db | Un servidor de base de datos (proceso aparte) |
| Instalación | Ninguna, viene con Python | Requiere instalar/administrar un servidor |
| Concurrencia | Limitada (un escritor a la vez) | Alta, pensada para muchos usuarios simultáneos |
| Tipos de datos | Flexibles (menos estrictos) | Estrictos, con muchos tipos avanzados |
| Cuándo usarla | Prototipos, apps chicas, tests, embebida en la app | Producción, apps con tráfico real |

Una gran ventaja práctica: el SQL que escribas en SQLite para este notebook es, en su gran mayoría, el mismo SQL que usarías en PostgreSQL — la sintaxis básica (`SELECT`, `JOIN`, etc.) es prácticamente idéntica.

Documentación oficial: [SQLite](https://www.sqlite.org/docs.html) · [PostgreSQL](https://www.postgresql.org/docs/) · [módulo sqlite3 de Python](https://docs.python.org/3/library/sqlite3.html)

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')  # base de datos en memoria, se pierde al cerrar la sesion
cur = conn.cursor()

cur.executescript('''
CREATE TABLE cliente (
    id INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL
);

CREATE TABLE producto (
    id INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    precio REAL NOT NULL,
    stock INTEGER NOT NULL DEFAULT 0
);

CREATE TABLE pedido (
    id INTEGER PRIMARY KEY,
    cliente_id INTEGER NOT NULL REFERENCES cliente(id),
    producto_id INTEGER NOT NULL REFERENCES producto(id),
    cantidad INTEGER NOT NULL,
    fecha TEXT NOT NULL
);
''')
conn.commit()
print('Esquema creado')

## 3. `INSERT`: cargar datos

```sql
INSERT INTO cliente (nombre, email) VALUES ('Ana Perez', 'ana@mail.com');
```

In [ ]:
cur.executemany(
    'INSERT INTO cliente (nombre, email) VALUES (?, ?)',
    [
        ('Ana Perez', 'ana@mail.com'),
        ('Bruno Diaz', 'bruno@mail.com'),
        ('Carla Ruiz', 'carla@mail.com'),
    ]
)

cur.executemany(
    'INSERT INTO producto (nombre, precio, stock) VALUES (?, ?, ?)',
    [
        ('Notebook Lenovo', 599.0, 10),
        ('Mouse inalambrico', 15.5, 100),
        ('Teclado mecanico', 45.0, 30),
    ]
)

cur.executemany(
    'INSERT INTO pedido (cliente_id, producto_id, cantidad, fecha) VALUES (?, ?, ?, ?)',
    [
        (1, 1, 1, '2026-01-05'),
        (1, 2, 2, '2026-01-05'),
        (2, 3, 1, '2026-02-10'),
        (3, 1, 1, '2026-03-01'),
    ]
)
conn.commit()
print('Datos cargados')

## 4. `SELECT`: consultar datos

```sql
SELECT nombre, precio FROM producto WHERE precio > 20 ORDER BY precio DESC LIMIT 5;
```

- `WHERE` filtra filas.
- `ORDER BY` ordena el resultado.
- `LIMIT` corta la cantidad de filas.
- Funciones de agregación (`COUNT`, `SUM`, `AVG`) junto con `GROUP BY` resumen datos por grupo.

In [ ]:
cur.execute('SELECT nombre, precio FROM producto WHERE precio > 20 ORDER BY precio DESC')
for fila in cur.fetchall():
    print(fila)

### Ejercicio 1 — `SELECT` con agregación

Escribí una consulta que devuelva, para cada `producto_id`, la cantidad total pedida (`SUM(cantidad)`), usando `GROUP BY`.

<details>
<summary>💡 Ver solución</summary>

```python
cur.execute('''
    SELECT producto_id, SUM(cantidad) AS total_pedido
    FROM pedido
    GROUP BY producto_id
''')
for fila in cur.fetchall():
    print(fila)
```

</details>

## 5. `UPDATE` y `DELETE`

```sql
UPDATE producto SET stock = stock - 1 WHERE id = 1;
DELETE FROM pedido WHERE id = 4;
```

**Siempre** con `WHERE` — sin filtro, `UPDATE`/`DELETE` afectan **todas** las filas de la tabla.

### Ejercicio 2 — Actualizar stock tras una venta

Escribí el código para: (1) descontar del `stock` del producto `1` la cantidad `1` (una venta), y (2) verificar el nuevo stock con un `SELECT`.

<details>
<summary>💡 Ver solución</summary>

```python
cur.execute('UPDATE producto SET stock = stock - ? WHERE id = ?', (1, 1))
conn.commit()

cur.execute('SELECT stock FROM producto WHERE id = 1')
print(cur.fetchone())
```

</details>

## 6. `JOIN`: combinar tablas

| Tipo | Devuelve |
|---|---|
| `INNER JOIN` | Solo filas que tienen coincidencia en ambas tablas |
| `LEFT JOIN` | Todas las filas de la tabla izquierda, con NULL donde no hay coincidencia en la derecha |

```sql
SELECT c.nombre, p.nombre AS producto, pe.cantidad
FROM pedido pe
INNER JOIN cliente c ON pe.cliente_id = c.id
INNER JOIN producto p ON pe.producto_id = p.id;
```

In [ ]:
cur.execute('''
    SELECT c.nombre, p.nombre AS producto, pe.cantidad
    FROM pedido pe
    INNER JOIN cliente c ON pe.cliente_id = c.id
    INNER JOIN producto p ON pe.producto_id = p.id
''')
for fila in cur.fetchall():
    print(fila)

### Ejercicio 3 — `LEFT JOIN`

Escribí una consulta con `LEFT JOIN` que liste todos los clientes junto con sus pedidos, incluyendo a los clientes que todavía no hicieron ningún pedido (si los hubiera).

<details>
<summary>💡 Ver solución</summary>

```python
cur.execute('''
    SELECT c.nombre, pe.id AS pedido_id, pe.cantidad
    FROM cliente c
    LEFT JOIN pedido pe ON pe.cliente_id = c.id
''')
for fila in cur.fetchall():
    print(fila)
```

</details>

## Mini-proyecto: reporte de ventas

Sobre el mismo esquema (`cliente`, `producto`, `pedido`):

1. Escribí una consulta que devuelva el gasto total por cliente (cantidad × precio, sumado), ordenado de mayor a menor.
2. Escribí una consulta que devuelva el producto más vendido (por cantidad total).
3. Insertá un nuevo cliente sin pedidos, y confirmá con un `LEFT JOIN` que aparece igual en el reporte, con cantidad `NULL`.

**Entregable:** las 3 consultas y su resultado.

---

**Seguís en:** *Colab 2 — ORM con SQLAlchemy y aplicación práctica de e-commerce*